# The first annotation using majority voting

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

## concat all

In [ ]:
data = sc.read_h5ad('./processed/rna/raw/data.h5ad')
dat = data[data.obs['dataset'] != 'Hu_BioRxiv_2023']
dat

In [ ]:
del data

In [ ]:
dat

In [ ]:
new = sc.read_h5ad('./processed/rna/raw/new.h5ad')
new

In [ ]:
new.obs = new.obs[dat.obs.columns.tolist()]

In [ ]:
all = pd.concat([dat.obs, new.obs], axis=0)
all

In [ ]:
all['cellType_raw_level2'].value_counts()

In [ ]:
all.to_csv('./all_8.csv')

## compare origin and pred

In [ ]:
all = pd.read_csv('./all_8.csv', index_col=0)
all

In [ ]:
all = all[all['cellType_raw_level2'] != 'notAvailable']
all

In [ ]:
all.columns

In [ ]:
df = all[['cellType_scarch_level2',
       'cellType_actinn_level2', 'cellType_celltypist_level2',
       'cellType_scmap_level2', 'cellType_singlecellnet_level2',
       'cellType_scpred_level2', 'cellType_singler_level2',
       'cellType_chetah_level2','cellType_raw_level2']]

In [ ]:
def calculate_similarity_ratio_all(df):
    last_col = df.iloc[:, -1]
    similarity_ratios = {}
    for col in df.columns[:-1]:  # Iterate over the first n-1 columns
        matches = (df[col] == last_col).sum()
        similarity_ratios[col] = matches / len(df)
    return similarity_ratios

In [ ]:
calculate_similarity_ratio_all(df)

## majority voting

In [ ]:
all = pd.read_csv('./all_8.csv', index_col=0)
all

In [ ]:
all.columns

In [ ]:
df = all[['cellType_actinn_level2', 'cellType_celltypist_level2',
       'cellType_singlecellnet_level2',
       'cellType_scpred_level2', 'cellType_singler_level2']]
df

In [ ]:
# Vectorized majority voting
def majority_vote_vectorized(df):
    # Count occurrences of each value within each row
    mode_counts = df.apply(lambda row: row.value_counts(), axis=1).fillna(0)

    # Find the most frequent value and its count
    max_counts = mode_counts.max(axis=1)
    majority_values = mode_counts.idxmax(axis=1)

    # Determine validity from the occurrence count
    result = majority_values.where(max_counts >= 3, "notAvailable")
    return result

In [ ]:
res = pd.DataFrame(majority_vote_vectorized(df))

In [ ]:
res[0].value_counts()

In [ ]:
res[0].value_counts()

In [ ]:
res.to_csv('./5_3_level2.csv')

## Consensus ratio

In [ ]:
all = pd.read_csv('./all_8.csv', index_col=0)
all

In [ ]:
df = all[['cellType_actinn_level2', 'cellType_celltypist_level2',
       'cellType_singlecellnet_level2',
       'cellType_scpred_level2', 'cellType_singler_level2']]
df

In [ ]:
l2 = pd.read_csv('5_3_level2.csv', index_col=0)
l2

In [ ]:
l2['0'].value_counts()

In [ ]:
l2.index.tolist() == all.index.tolist()

In [ ]:
df['mv'] = l2['0'].tolist()

In [ ]:
df

In [ ]:
def calculate_similarity_ratio_all(df):
    last_col = df.iloc[:, -1]
    similarity_ratios = {}
    for col in df.columns[:-1]:  # Iterate over the first n-1 columns
        matches = (df[col] == last_col).sum()
        similarity_ratios[col] = matches / len(df)
    return similarity_ratios

In [ ]:
calculate_similarity_ratio_all(df)

In [ ]:
24*0.8510280376108648

In [ ]:
# Compare with the original annotations
l2['cellType_raw_level2'] = all['cellType_raw_level2'].tolist()
l2 = l2[all['cellType_raw_level2'] != 'notAvailable']
l2

In [ ]:
calculate_similarity_ratio_all(l2)

## 3 method agree

In [ ]:
all = pd.read_csv('./all_8.csv', index_col=0)
all = all[all['cellType_raw_level2'] != 'notAvailable']
all

In [ ]:
test = all[['cellType_celltypist_level2','cellType_singlecellnet_level2','cellType_scpred_level2']]
test

In [ ]:
# Vectorized implementation of majority voting
def majority_vote_vectorized(df):
    # Count the frequency of each value per row
    mode_counts = df.apply(lambda row: row.value_counts(), axis=1).fillna(0)

    # Find the most frequent value and its occurrence count
    max_counts = mode_counts.max(axis=1)
    majority_values = mode_counts.idxmax(axis=1)

    # Determine validity based on the occurrence count
    # (Retain the value if it appears at least twice, otherwise mark as "notAvailable")
    result = majority_values.where(max_counts >= 2, "notAvailable")
    return result

In [ ]:
# level2
res = pd.DataFrame(majority_vote_vectorized(test))
res[0].value_counts()

In [ ]:
res['cellType_raw_level2'] = all['cellType_raw_level2'].tolist()
res

In [ ]:
def calculate_similarity_ratio_all(df):
    last_col = df.iloc[:, -1]
    similarity_ratios = {}
    for col in df.columns[:-1]:  
        matches = (df[col] == last_col).sum()
        similarity_ratios[col] = matches / len(df)
    return similarity_ratios

In [ ]:
calculate_similarity_ratio_all(res)

# The second annotation using transfer

In [ ]:
import scanpy as sc
import pandas as pd

## Read data, add mv results

In [ ]:
data = sc.read_h5ad('./processed/rna/data_new.h5ad')
data

In [ ]:
mv = pd.read_csv('./processed/ann/5_3_level2.csv', index_col=0)
mv

In [ ]:
data.obs_names.tolist() == mv.index.tolist()

In [ ]:
data.obs['mv'] = mv['0'].tolist()

In [ ]:
dat = data[data.obs['mv'] != 'notAvailable']
dat

In [ ]:
dat.obs['cellType_raw'].value_counts()

In [ ]:
1476837-324948

In [ ]:
dat = data[data.obs['mv'] == 'notAvailable']
dat

In [ ]:
dat.obs['cellType_raw'].value_counts()

In [ ]:
165029-24176

In [ ]:
import matplotlib.pyplot as plt

# Data
sizes = [140853, 24176, 1151889, 324948]
labels = ['165029', '24176', '1476837', '324948']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']  # Four colors
explode = (0, 0, 0.1, 0)  # Highlight the third slice (1476837)

# Draw the pie chart
plt.figure(figsize=(8, 8))
plt.pie(sizes,  labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Pie Chart of Four Values')
plt.axis('equal')  # Use equal aspect ratio to keep the pie circular

# Save and display
plt.savefig('pie_chart_four_values.pdf')
plt.show()

print("Pie chart saved as 'pie_chart_four_values.png'")

## train model

In [ ]:
import celltypist

In [ ]:
sc.pp.normalize_total(dat, target_sum=1e4)
sc.pp.log1p(dat)
sc.pp.highly_variable_genes(dat, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample')

In [ ]:
adata = dat[:, dat.var.highly_variable]
adata

In [ ]:
# new_model = celltypist.train(adata, 
#                              labels = 'mv', 
#                              check_expression = False,
#                              use_SGD = True,
#                              mini_batch = True)

In [ ]:
new_model.write('./model.pkl')

## annotate cell types

In [ ]:
# new_model = models.Model.load('./model.pkl')

In [ ]:
da = data[data.obs['mv'] == 'notAvailable']
da

In [ ]:
sc.pp.normalize_total(da, target_sum=1e4)
sc.pp.log1p(da)

In [ ]:
predictions = celltypist.annotate(da, model = new_model, majority_voting=True)

In [ ]:
da_a = predictions.to_adata()

In [ ]:
da_a

In [ ]:
dd = da_a.obs[['predicted_labels']]
dd

In [ ]:
dd.to_csv('./two_transfer.csv')

## Get all annotation

In [ ]:
data = sc.read_h5ad('./processed/rna/data_new.h5ad')
data

In [ ]:
mv = pd.read_csv('./5_3_level2.csv', index_col=0)
mv = mv[mv['0'] != 'notAvailable']
mv

In [ ]:
two = pd.read_csv('./two_transfer.csv', index_col=0)
two.columns = ['0']
two

In [ ]:
all = pd.concat([mv, two], axis=0)
all = all.reindex(data.obs_names.tolist())
all

In [ ]:
classification_dict = {
    'Granulocytes': 'Monocytes',
    'Promyelocytes': 'Monocytes',
    'DC precursor': 'DC',
    'ETP': 'T cells',
    'Megakaryocytes/platelets': 'Early MK',
    'Megakaryocyte precursor': 'Early MK'
}

In [ ]:
all.replace(classification_dict, inplace=True)

In [ ]:
all['0'].value_counts()

In [ ]:
data.obs_names.tolist() == all.index.tolist()

In [ ]:
data.obs['final_ann'] = all['0'].tolist()

In [ ]:
data.obs['final_ann'].value_counts()

In [ ]:
data

In [ ]:
data.write_h5ad('./processed/adata.h5ad')

## Compare the original annotation with the individual ratio plots of all cells

In [ ]:
data = sc.read_h5ad('./processed/adata.h5ad')
data

In [ ]:
data

In [ ]:
data.obs['cellType_raw_level2'].value_counts()

In [ ]:
classification_dict = {
    'Double-negative thymocytes': 'T cells',
    'Megakaryocytes/platelets': 'Early MK',
    'Megakaryocyte precursor': 'Early MK',
    'Granulocytes': 'Monocytes',
    'DC precursor': 'DC'
}

In [ ]:
other = data.obs[['cellType_raw_level2']]
other

In [ ]:
other.replace(classification_dict, inplace=True)

In [ ]:
other['cellType_raw_level2'].value_counts()

In [ ]:
data.obs['cellType_raw_level2'] = other['cellType_raw_level2'].tolist()

In [ ]:
data.write_h5ad('./processed/adata.h5ad')

In [ ]:
other['final_ann'] = data.obs['final_ann'].tolist()
other

In [ ]:
# Initialize an empty dictionary to store results
results = {}

# Get all unique categories from the 'final_ann' column
categories = other['final_ann'].unique()

# Iterate through each category to calculate ratios
for category in categories:
    # Filter data for the current category
    subset = other[other['final_ann'] == category]
    
    # Total count of entries in this category
    total = len(subset)
    
    # Calculate the ratio of matches (where 'cellType_raw_level2' matches 'final_ann')
    match_ratio = (subset['cellType_raw_level2'] == subset['final_ann']).sum() / total
    
    # Calculate the ratio of entries marked as 'notAvailable'
    not_available_ratio = (subset['cellType_raw_level2'] == 'notAvailable').sum() / total
    
    # Calculate the remaining ratio (i.e., 1 - match ratio - notAvailable ratio)
    other_ratio = 1 - match_ratio - not_available_ratio
    
    # Store the calculated metrics in the results dictionary
    results[category] = {
        'match_ratio': match_ratio,
        'not_available_ratio': not_available_ratio,
        'other_ratio': other_ratio
    }

# Convert the results dictionary to a DataFrame for easier viewing
result_df = pd.DataFrame(results).T
result_df.index.name = 'Category'

# Print the final DataFrame
print(result_df)

In [ ]:
result_df['Category'] = result_df.index.tolist()
result_df

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df = result_df.sort_values(by='match_ratio', ascending=False)
df

In [ ]:
# Plot the stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))

# Plot each segment
ax.bar(df['Category'], df['match_ratio'], label='Match Ratio', color='#32cd32')
ax.bar(df['Category'], df['not_available_ratio'], 
       bottom=df['match_ratio'], label='Not Available Ratio', color='#4682b4')
ax.bar(df['Category'], df['other_ratio'], 
       bottom=df['match_ratio'] + df['not_available_ratio'], label='Other Ratio', color='#ff0000')

# Add title and axis labels
# ax.set_title('Proportion of Ratios by Category', fontsize=16)
# ax.set_xlabel('Category', fontsize=12)
# ax.set_ylabel('Proportion', fontsize=12)
ax.set_ylim(0, 1)  # Set the y-axis limits from 0 to 1
ax.legend(title='Ratio Type')

# Display grid lines
# ax.grid(axis='y', linestyle='--', alpha=0.7)

# Save as PDF
plt.tight_layout()
plt.savefig('stacked_bar_chart.pdf')

# Show the plot
plt.show()

In [ ]:
# total
# notAvailable      
349597/1641866

In [ ]:
def calculate_similarity_ratio_all(df):
    last_col = df.iloc[:, -1]
    similarity_ratios = {}
    for col in df.columns[:-1]:  
        matches = (df[col] == last_col).sum()
        similarity_ratios[col] = matches / len(df)
    return similarity_ratios

In [ ]:
calculate_similarity_ratio_all(other)

In [ ]:
1-0.212926633476788-0.6844839956488532

In [ ]:
24*0.6844839956488532

In [ ]:
24*0.212926633476788

In [ ]:
24*0.10258937087435882

## The ratio of all cells

In [ ]:
adata = sc.read_h5ad('./processed/adata.h5ad')
adata

In [ ]:
final_ann = pd.DataFrame(adata.obs['final_ann'].value_counts())
final_ann['final'] = final_ann.index.tolist()
final_ann.sort_values(by='final', inplace=True)
final_ann['color'] = ['#1f77b4',
               '#ff7f0e',
               '#279e68',
               '#d62728',
               '#aa40fc',
               '#8c564b',
               '#e377c2',
               '#b5bd61',
               '#17becf',
               '#aec7e8',
               '#ffbb78',
               '#98df8a',
               '#ff9896',
               '#c5b0d5']
final_ann.sort_values(by='count', inplace=True, ascending=False)
final_ann[['count', 'color']]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

data = {
    'Cell Type': ['T cells', 'Monocytes', 'ILC', 'B cells', 'HSC/MPP', 'Erythroid', 'pDC', 
                  'DC', 'Plasma cells', 'Mast cells', 'Early MK', 'Macrophages', 
                  'B-cell lineage', 'Cycling cells'],
    'Count': [1115932, 169481, 106651, 103190, 88479, 30220, 8876, 7078, 4796, 
              2912, 2693, 624, 529, 405],
    'Color': ['#ff9896', '#ffbb78', '#b5bd61', '#1f77b4', '#e377c2', '#8c564b', '#c5b0d5', 
              '#d62728', '#98df8a', '#aec7e8', '#aa40fc', '#17becf', '#ff7f0e', '#279e68']
}

df = pd.DataFrame(data)

plt.figure(figsize=(8, 8))
plt.pie(df['Count'], labels=df['Cell Type'], autopct='%1.1f%%', startangle=90, colors=df['Color'])
plt.title('Cell Type Distribution')
plt.axis('equal')  
plt.savefig('cell_type_distribution_1.pdf', format='pdf')
plt.show()

In [ ]:
30220/1641866

In [ ]:
final_ann = pd.DataFrame(adata.obs['cellType_raw_level2'].value_counts())
final_ann['final'] = final_ann.index.tolist()
final_ann.sort_values(by='final', inplace=True)
final_ann

In [ ]:
final_ann['color'] = ['#1f77b4',
               '#ff7f0e',
               '#279e68',
               '#d62728',
               '#aa40fc',
               '#8c564b',
               '#e377c2',
               '#b5bd61',
               '#17becf',
               '#aec7e8',
               '#ffbb78',
               '#98df8a',
               '#ff9896',
               '#cccecf',
               '#c5b0d5']
final_ann.sort_values(by='count', inplace=True, ascending=False)
final_ann[['count', 'color']]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Data
data = {
    'Cell Type': ['T cells', 'notAvailable', 'HSC/MPP', 'Monocytes', 'ILC', 'B cells', 'Erythroid', 
                  'Early MK', 'DC', 'B-cell lineage', 'pDC', 'Plasma cells', 'Cycling cells', 
                  'Macrophages', 'Mast cells'],
    'Count': [808535, 349597, 122500, 99440, 89889, 75188, 46373, 14000, 9264, 7959, 7048, 
              4250, 3427, 2702, 1694],
    'Color': ['#ff9896', '#cccecf', '#e377c2', '#ffbb78', '#b5bd61', '#1f77b4', '#8c564b', '#aa40fc',
              '#d62728', '#ff7f0e', '#c5b0d5', '#98df8a', '#279e68', '#17becf', '#aec7e8']
}

# Create a DataFrame
df = pd.DataFrame(data)

# Draw the pie chart
plt.figure(figsize=(8, 8))
plt.pie(df['Count'], labels=df['Cell Type'], autopct='%1.1f%%', startangle=90, colors=df['Color'])
plt.title('Cell Type Distribution')
plt.axis('equal')  # Use equal aspect ratio to keep the pie circular
plt.savefig('cell_type_distribution_2.pdf', format='pdf')
plt.show()

In [ ]:
14000/1641866

## marker gene

In [ ]:
data = sc.read_h5ad('./processed/adata.h5ad')
data

In [ ]:
sc.pp.normalize_total(data, target_sum=1e4)
sc.pp.log1p(data)

In [ ]:
data.obs['final_ann'].value_counts()

In [ ]:
# Sort by distinction
marker_genes_dict = {
    "B cells": ["BANK1","MS4A1", "CD79A"],
    "B-cell lineage": ["DNTT", "MME", "VPREB1"], 
    "Cycling cells": ["MKI67", "TOP2A", "CDK1"],
    "DC": ["HLA-DRA",  "CST3", "ITGAX"],
    "Early MK": [ "ITGA2B", "ITGB3","PF4"],
    "Erythroid": [ "HBA1", "HBA2", "ALAS2"],
    "HSC/MPP": [ "PROM1", "SPINK2", "CD34"],
    "ILC": ["CD247", "GNLY", "NKG7"], 
    "Macrophages": ["RBM47", "DOCK4", "PLAUR"],
    "Mast cells": ["KIT", "CPA3","TPSAB1"],
    "Monocytes": [ "VCAN", "LYZ", "S100A9"],
    "Plasma cells": ["JCHAIN", "MZB1","XBP1"],
    "T cells": ["IL7R", "ANK3","LEF1"],
    "pDC": ["TCF4", "IL3RA", "CLEC4C"]
}

In [ ]:
adata = sc.read_h5ad('./dendrogram.h5ad')
adata

In [ ]:
data.uns['dendrogram_final_ann'] = adata.uns['dendrogram_final_ann']

In [ ]:
sc.pl.dotplot(data, marker_genes_dict, groupby='final_ann', dendrogram=True, save='_marker') 

In [ ]:
# combine DEG

In [ ]:
sc.tl.rank_genes_groups(data, "final_ann", method="t-test")

In [ ]:
pd.DataFrame(data.uns["rank_genes_groups"]["names"]).head(20)[['T cells']]

In [ ]:
sc.pl.dotplot(data, [
    'IL7R',
    'THEMIS',
    'LEF1',
    'TCF7',
    'ITK',
    'SKAP1',
    'TRAC',
    'CD6',
    'PRKCA',
    'ANK3'
], groupby='final_ann')

## dendrogram

In [ ]:
# adata = sc.read_h5ad('./processed/adata.h5ad')
# sc.pp.calculate_qc_metrics(adata,inplace=True)
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample')
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, svd_solver='arpack')

# sce.pp.harmony_integrate(adata, 'sample')
# adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

# sc.tl.dendrogram(adata, "final_ann")

# adata.write_h5ad('./dendrogram.h5ad')

In [ ]:
data = sc.read_h5ad('./dendrogram.h5ad')
data

In [ ]:
sc.pl.dendrogram(data, 'final_ann')